In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
"""
SHAP-pruned feature set: retrain Ridge / RF / XGBoost per horizon
using only the top-N SHAP features.

  - Loads data + full-feature matrix ONCE and caches it in globals,
    so re-running a horizon cell doesn't redo the Hopsworks read/median-fill.
  - Gives each horizon (24h / 48h / 72h) its OWN cell. Run only
    the one you want. Each cell prints its RMSE/MAE/R2 as soon as it's done
    for a model — not after everything finishes.
  - SHAP ranking is cached per-horizon after first load/compute, so it's never recomputed
    twice in the same session.
  - Results are appended to shap_pruned_results.csv incrementally (as before)
    so partial progress (e.g. just 24h) is never lost even if you stop.

Setup on Kaggle:
    1. Add-ons -> Secrets -> add HOPSWORKS_API_KEY
    2. pip install hopsworks xgboost shap
"""

In [1]:
import argparse
import os
import sys
import time
from typing import Dict, Tuple

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor

In [2]:
# ── Install deps (Kaggle base image doesn't have hopsworks) ────────────────
import subprocess

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "hopsworks", "shap"], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "protobuf>=5.28,<6"],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pyopenssl"],
    check=True,
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.2/44.2 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 258.6/258.6 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.3/45.3 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 105.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.7/45.7 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 88.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.9/91.9 kB 4.7 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
grpcio-tools 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 4.25.9 which is incompatible.
sigstore 4.3.0 requires cryptography<49,>=42, but you have cryptography 50.0.0 which is incompatible.
a2a-sdk 0.3.26 requires protobuf>=5.29.5, but you have protobuf 4.25.9 which is incompatible.
tensorflow 2.20.0 requires protobuf>=5.28.0, but you have protobuf 4.25.9 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 4.25.9 which is incompatible.
pydrive2 1.21.3 requires cryptography<44, but you have cryptography 50.0.0 which is incompatible.
pyopenssl 24.2.1 requires cryptography<44,>=4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.5/320.5 kB 18.9 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
hopsworks 5.0.4 requires protobuf<5.0.0,>=4.25.4, but you have protobuf 5.29.6 which is incompatible.


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 4.0 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sigstore 4.3.0 requires cryptography<49,>=42, but you have cryptography 50.0.0 which is incompatible.
pydrive2 1.21.3 requires cryptography<44, but you have cryptography 50.0.0 which is incompatible.
pydrive2 1.21.3 requires pyOpenSSL<=24.2.1,>=19.1.0, but you have pyopenssl 26.4.0 which is incompatible.


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', '--upgrade', 'pyopenssl'], returncode=0)

In [3]:
# ========================================================================
# Config
# ========================================================================

FEATURE_GROUP_NAME = "aqi_features"
FEATURE_GROUP_VERSION = 1
HORIZONS = ["24h", "48h", "72h"]

TRAIN_TEST_SPLIT = 0.8
CV_N_SPLITS = 5
HPT_CV_FOLDS = 5
HPT_N_ITER = 20

TOP_N = 15  # number of SHAP-ranked features to keep per horizon

SHAP_CSV_TEMPLATE = "/kaggle/working/shap_importance_{h}.csv"
LSTM_RESULTS_CSV = "/kaggle/working/lstm_results.csv"
PRUNED_RESULTS_CSV = "/kaggle/working/shap_pruned_results.csv"

# "timestamp" is excluded explicitly and asserted against below — it must
# never reach any model as a feature, pruned or not.
NON_FEATURE_COLS = {
    "timestamp", "has_target",
    "target_aqi_24h", "target_aqi_48h", "target_aqi_72h",
}

In [4]:
import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("shap_pruned_kaggle")


In [5]:
# ========================================================================
# Hopsworks connection
# ========================================================================

def get_feature_store():
    import hopsworks

    api_key = None
    try:
        from kaggle_secrets import UserSecretsClient
        api_key = UserSecretsClient().get_secret("HOPSWORKS_API_KEY")
    except Exception:
        api_key = os.environ.get("HOPSWORKS_API_KEY")

    if not api_key:
        raise RuntimeError(
            "No Hopsworks API key found. Add it under Add-ons -> Secrets as "
            "HOPSWORKS_API_KEY, or set the HOPSWORKS_API_KEY environment variable."
        )

    project = hopsworks.login(api_key_value=api_key)
    return project.get_feature_store()


In [6]:
def chronological_holdout_split(df: pd.DataFrame, train_fraction: float):
    split_idx = int(len(df) * train_fraction)
    return df.iloc[:split_idx].reset_index(drop=True), df.iloc[split_idx:].reset_index(drop=True)


def walk_forward_splits(n_rows: int, n_splits: int):
    tscv = TimeSeriesSplit(n_splits=n_splits)
    return list(tscv.split(np.arange(n_rows)))

In [7]:
def regression_metrics(y_true, y_pred) -> Dict[str, float]:
    return {
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "r2": float(r2_score(y_true, y_pred)),
    }

In [8]:
def append_results(rows: list):
    new_df = pd.DataFrame(rows)
    if os.path.exists(PRUNED_RESULTS_CSV):
        existing = pd.read_csv(PRUNED_RESULTS_CSV)
        new_df = pd.concat([existing, new_df], ignore_index=True)
    new_df.to_csv(PRUNED_RESULTS_CSV, index=False)
    logger.info("Wrote %d rows to %s (%d total)", len(rows), PRUNED_RESULTS_CSV, len(new_df))


In [9]:
def get_feature_columns(df: pd.DataFrame) -> list:
    feature_cols = [c for c in df.columns if c not in NON_FEATURE_COLS]
    assert "timestamp" not in feature_cols, "timestamp leaked into feature columns"
    return feature_cols

In [10]:
def split_xy(df: pd.DataFrame, feature_cols: list) -> Tuple[pd.DataFrame, Dict[str, pd.Series]]:
    X = df[feature_cols].copy()
    y = {h: df[f"target_aqi_{h}"] for h in HORIZONS if f"target_aqi_{h}" in df.columns}
    return X, y


In [11]:
def load_or_compute_shap_ranking(h: str, X_train: pd.DataFrame, y_train: pd.Series,
                                  X_holdout: pd.DataFrame, feature_cols: list) -> pd.DataFrame:
    path = SHAP_CSV_TEMPLATE.format(h=h)
    if os.path.exists(path):
        logger.info("  [SHAP] loading precomputed ranking from %s", path)
        return pd.read_csv(path)

    logger.warning("  [SHAP] %s not found — recomputing from a fresh XGBoost fit "
                    "(run train_classical_kaggle_improved.py first to avoid this)", path)
    import shap

    param_dist = {
        "n_estimators": [200, 400, 600, 800],
        "max_depth": [4, 6, 8, 10],
        "learning_rate": [0.01, 0.03, 0.05, 0.1],
        "subsample": [0.7, 0.8, 0.9, 1.0],
        "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
        "min_child_weight": [1, 3, 5, 7],
    }
    cv_splits = walk_forward_splits(len(X_train), n_splits=HPT_CV_FOLDS)
    search = RandomizedSearchCV(
        XGBRegressor(objective="reg:squarederror", random_state=42, n_jobs=1, tree_method="hist"),
        param_distributions=param_dist, n_iter=HPT_N_ITER, cv=cv_splits,
        scoring="neg_root_mean_squared_error", n_jobs=-1, random_state=42, verbose=0,
    )
    search.fit(X_train, y_train)
    explainer = shap.TreeExplainer(search.best_estimator_)
    shap_values = explainer.shap_values(X_holdout)
    mean_abs_shap = np.abs(shap_values).mean(axis=0)
    importance = pd.DataFrame({
        "feature": feature_cols, "mean_abs_shap": mean_abs_shap,
    }).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
    importance.to_csv(path, index=False)
    return importance

In [12]:
def fit_ridge(X_train, y_train, X_holdout, quick: bool = False):
    pipe = Pipeline([("scaler", StandardScaler()), ("ridge", Ridge())])
    alpha_grid = [0.1, 1.0, 10.0] if quick else [0.001, 0.01, 0.1, 1.0, 10.0, 30.0, 100.0, 300.0]
    cv_splits = walk_forward_splits(len(X_train), n_splits=CV_N_SPLITS)
    search = GridSearchCV(pipe, param_grid={"ridge__alpha": alpha_grid}, cv=cv_splits,
                           scoring="neg_root_mean_squared_error", n_jobs=-1)
    search.fit(X_train, y_train)
    return search.best_estimator_, search.best_estimator_.predict(X_holdout)

In [13]:
def fit_rf(X_train, y_train, X_holdout, quick: bool = False):
    if quick:
        param_grid = {"n_estimators": [50], "max_depth": [10]}
    else:
        param_grid = {"n_estimators": [100, 200, 400], "max_depth": [10, 20, 30, None], "min_samples_leaf": [1, 2, 4]}
    cv_splits = walk_forward_splits(len(X_train), n_splits=CV_N_SPLITS)
    search = GridSearchCV(RandomForestRegressor(random_state=42, n_jobs=-1),
                           param_grid=param_grid, cv=cv_splits,
                           scoring="neg_root_mean_squared_error", n_jobs=-1)
    search.fit(X_train, y_train)
    return search.best_estimator_, search.best_estimator_.predict(X_holdout)

In [14]:
def fit_xgb(X_train, y_train, X_holdout, quick: bool = False):
    n_iter = 5 if quick else HPT_N_ITER
    param_dist = {
        "n_estimators": [200, 400, 600, 800],
        "max_depth": [4, 6, 8, 10],
        "learning_rate": [0.01, 0.03, 0.05, 0.1],
        "subsample": [0.7, 0.8, 0.9, 1.0],
        "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
        "min_child_weight": [1, 3, 5, 7],
    }
    cv_splits = walk_forward_splits(len(X_train), n_splits=HPT_CV_FOLDS)
    search = RandomizedSearchCV(
        XGBRegressor(objective="reg:squarederror", random_state=42, n_jobs=1, tree_method="hist"),
        param_distributions=param_dist, n_iter=n_iter, cv=cv_splits,
        scoring="neg_root_mean_squared_error", n_jobs=-1, random_state=42, verbose=0,
    )
    search.fit(X_train, y_train)
    return search.best_estimator_, search.best_estimator_.predict(X_holdout)

In [15]:
MODEL_FITTERS = [("Ridge_pruned", fit_ridge), ("RF_pruned", fit_rf), ("XGBoost_pruned", fit_xgb)]

# Session-level caches so re-running a horizon cell doesn't redo work that's
# already done. _SHAP_CACHE holds the ranking per horizon so a re-run of the
# same horizon cell (e.g. after changing TOP_N) reuses it instead of
# reloading/recomputing.
_DATA_CACHE = {}
_SHAP_CACHE = {}

In [17]:
def load_training_data() -> pd.DataFrame:
    logger.info("Reading aqi_features from Hopsworks")
    fs = get_feature_store()
    fg = fs.get_feature_group(FEATURE_GROUP_NAME, version=FEATURE_GROUP_VERSION)
    df = fg.read()
    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
    df = df.sort_values("timestamp").reset_index(drop=True)
    df = df[df["has_target"] == 1].copy()
    logger.info("Rows with full targets: %d", len(df))
    return df

In [16]:
def load_data():
    df = load_training_data()
    train_df, holdout_df = chronological_holdout_split(df, train_fraction=TRAIN_TEST_SPLIT)
    feature_cols = get_feature_columns(df)
    logger.info("Full feature set: %d columns (timestamp excluded)", len(feature_cols))

    X_train_full, y_train_dict = split_xy(train_df, feature_cols)
    X_holdout_full, y_holdout_dict = split_xy(holdout_df, feature_cols)
    feature_medians = X_train_full.median()
    X_train_full = X_train_full.fillna(feature_medians)
    X_holdout_full = X_holdout_full.fillna(feature_medians)

    _DATA_CACHE.update({
        "feature_cols": feature_cols,
        "X_train_full": X_train_full,
        "X_holdout_full": X_holdout_full,
        "y_train_dict": y_train_dict,
        "y_holdout_dict": y_holdout_dict,
    })
    logger.info("Cell 1 done — data cached. Now run cell_run_horizon('24h'), '48h', or '72h' — one at a time.")


In [19]:
load_data()

2026-08-08 19:16:47,809 [INFO] Reading aqi_features from Hopsworks
2026-08-08 19:16:48,117 [INFO] Initializing external client
2026-08-08 19:16:48,118 [INFO] Base URL: https://eu-west.cloud.hopsworks.ai:443
2026-08-08 19:16:51,307 [INFO] Python Engine initialized.



Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/42156


2026-08-08 19:16:57,233 [INFO] Rows with full targets: 18192
2026-08-08 19:16:57,235 [INFO] Full feature set: 34 columns (timestamp excluded)
2026-08-08 19:16:57,258 [INFO] Cell 1 done — data cached. Now run cell_run_horizon('24h'), '48h', or '72h' — one at a time.


Finished: Reading data from Hopsworks, using Hopsworks Feature Query Service (1.87s) 


In [20]:
def run_horizon(h: str, top_n: int = TOP_N, quick: bool = False):
    if not _DATA_CACHE:
        raise RuntimeError("Run cell1_load_data() first — data isn't cached yet.")
    if h not in HORIZONS:
        raise ValueError(f"h must be one of {HORIZONS}, got {h!r}")

    feature_cols = _DATA_CACHE["feature_cols"]
    X_train_full = _DATA_CACHE["X_train_full"]
    X_holdout_full = _DATA_CACHE["X_holdout_full"]
    y_train = _DATA_CACHE["y_train_dict"][h]
    y_holdout = _DATA_CACHE["y_holdout_dict"][h]

    logger.info("=== Horizon: %s (top %d SHAP features) ===", h, top_n)

    cache_key = (h, top_n)
    if cache_key in _SHAP_CACHE:
        ranking = _SHAP_CACHE[cache_key]
        logger.info("  [SHAP] reusing cached ranking for %s", h)
    else:
        ranking = load_or_compute_shap_ranking(h, X_train_full, y_train, X_holdout_full, feature_cols)
        _SHAP_CACHE[cache_key] = ranking
    top_features = ranking["feature"].head(top_n).tolist()
    assert "timestamp" not in top_features, "timestamp leaked into SHAP-selected features"
    logger.info("  Top %d features: %s", top_n, top_features)

    X_train = X_train_full[top_features]
    X_holdout = X_holdout_full[top_features]

    print("\n" + "=" * 70)
    print(f"HORIZON {h} — top {top_n} SHAP features")
    print("=" * 70)

    horizon_rows = []
    for model_name, fit_fn in MODEL_FITTERS:
        t0 = time.time()
        _, pred = fit_fn(X_train, y_train, X_holdout, quick=quick)
        fit_time = time.time() - t0
        m = regression_metrics(y_holdout, pred)

        row = {"model": model_name, "horizon": h.replace("h", ""), "split": "holdout", "n_features": top_n}
        row.update(m)
        horizon_rows.append(row)
        append_results([row])  # write to disk immediately — nothing is lost if you stop here
        # print right away, don't wait for the other models/horizons
        print(f"  {model_name:<16s} fit_time={fit_time:6.1f}s  RMSE={m['rmse']:.2f}  MAE={m['mae']:.2f}  R2={m['r2']:.3f}")

    print(f"\nHorizon {h} done. Results appended to {PRUNED_RESULTS_CSV}.")
    print("Run cell_run_horizon() for another horizon, or cell5_leaderboard() to compare progress so far.\n")
    return pd.DataFrame(horizon_rows)

In [21]:
run_horizon("24h")

2026-08-08 19:18:08,340 [INFO] === Horizon: 24h (top 15 SHAP features) ===
2026-08-08 19:18:08,342 [WARNING]   [SHAP] /kaggle/working/shap_importance_24h.csv not found — recomputing from a fresh XGBoost fit (run train_classical_kaggle_improved.py first to avoid this)
2026-08-08 19:22:06,010 [INFO]   Top 15 features: ['pm2_5', 'us_aqi', 'month_cos', 'aqi_change_rate_1h', 'aqi_lag_24h', 'rolling_30day_avg', 'pm10', 'relative_humidity_2m', 'pm2_5_lag_24h', 'pressure_msl', 'aqi_rolling_24h', 'wind_speed_10m', 'o3', 'rolling_30day_std', 'hour_sin']



HORIZON 24h — top 15 SHAP features


2026-08-08 19:22:06,327 [INFO] Wrote 1 rows to /kaggle/working/shap_pruned_results.csv (1 total)


  Ridge_pruned     fit_time=   0.3s  RMSE=10.69  MAE=7.35  R2=0.516


2026-08-08 19:46:35,223 [INFO] Wrote 1 rows to /kaggle/working/shap_pruned_results.csv (2 total)


  RF_pruned        fit_time=1468.9s  RMSE=11.28  MAE=7.72  R2=0.461


2026-08-08 19:48:22,143 [INFO] Wrote 1 rows to /kaggle/working/shap_pruned_results.csv (3 total)


  XGBoost_pruned   fit_time= 106.9s  RMSE=10.82  MAE=7.40  R2=0.504

Horizon 24h done. Results appended to /kaggle/working/shap_pruned_results.csv.
Run cell_run_horizon() for another horizon, or cell5_leaderboard() to compare progress so far.



,model,horizon,split,n_features,rmse,mae,r2
0,Ridge_pruned,24,holdout,15,10.687623,7.354269,0.516318
1,RF_pruned,24,holdout,15,11.279417,7.722688,0.461270
2,XGBoost_pruned,24,holdout,15,10.819855,7.395762,0.504275


In [22]:
run_horizon("48h")

2026-08-08 19:48:41,336 [INFO] === Horizon: 48h (top 15 SHAP features) ===
2026-08-08 19:48:41,337 [WARNING]   [SHAP] /kaggle/working/shap_importance_48h.csv not found — recomputing from a fresh XGBoost fit (run train_classical_kaggle_improved.py first to avoid this)
2026-08-08 19:52:03,475 [INFO]   Top 15 features: ['rolling_30day_avg', 'us_aqi', 'month_cos', 'pm2_5', 'pressure_msl', 'month', 'rolling_30day_std', 'aqi_change_rate_1h', 'day_of_week', 'month_sin', 'pm10', 'aqi_lag_72h', 'aqi_lag_24h', 'aqi_rolling_6h', 'wind_speed_10m']



HORIZON 48h — top 15 SHAP features


2026-08-08 19:52:03,740 [INFO] Wrote 1 rows to /kaggle/working/shap_pruned_results.csv (4 total)


  Ridge_pruned     fit_time=   0.3s  RMSE=14.68  MAE=10.35  R2=0.087


2026-08-08 20:13:42,575 [INFO] Wrote 1 rows to /kaggle/working/shap_pruned_results.csv (5 total)


  RF_pruned        fit_time=1298.8s  RMSE=14.62  MAE=10.18  R2=0.094


2026-08-08 20:15:13,780 [INFO] Wrote 1 rows to /kaggle/working/shap_pruned_results.csv (6 total)


  XGBoost_pruned   fit_time=  91.2s  RMSE=14.16  MAE=9.92  R2=0.150

Horizon 48h done. Results appended to /kaggle/working/shap_pruned_results.csv.
Run cell_run_horizon() for another horizon, or cell5_leaderboard() to compare progress so far.



,model,horizon,split,n_features,rmse,mae,r2
0,Ridge_pruned,48,holdout,15,14.680102,10.347948,0.086970
1,RF_pruned,48,holdout,15,14.623979,10.177178,0.093938
2,XGBoost_pruned,48,holdout,15,14.164905,9.915791,0.149931


In [ ]:
run_horizon("72h")

2026-08-08 20:15:21,947 [INFO] === Horizon: 72h (top 15 SHAP features) ===
2026-08-08 20:15:21,949 [WARNING]   [SHAP] /kaggle/working/shap_importance_72h.csv not found — recomputing from a fresh XGBoost fit (run train_classical_kaggle_improved.py first to avoid this)
2026-08-08 20:18:36,414 [INFO]   Top 15 features: ['rolling_30day_avg', 'month_cos', 'rolling_30day_std', 'us_aqi', 'pm2_5', 'pressure_msl', 'month_sin', 'month', 'aqi_lag_24h', 'aqi_change_rate_1h', 'so2', 'day_of_week', 'aqi_lag_72h', 'aqi_lag_48h', 'hour_cos']



HORIZON 72h — top 15 SHAP features


2026-08-08 20:18:36,750 [INFO] Wrote 1 rows to /kaggle/working/shap_pruned_results.csv (7 total)


  Ridge_pruned     fit_time=   0.3s  RMSE=15.55  MAE=11.05  R2=-0.024
